# 9차시 강사용 Notebook — 머신러닝 예측 모델 만들기

**시연 전 안내**: Orange3로 File → Tree → Test and Score를 먼저 보여준 뒤 이 노트북으로 넘어갑니다.

## 1단계. 데이터 읽고 결측치 처리하기

> ⚠️ **부호 규약 전환 — 9차시를 시작할 때 반드시 짚고 넘어간다.**
>
> 1~8차시의 `합격여부`는 `1`=합격, `-1`=불합격이었는데,
> 9·10차시의 `검사결과`는 **`0`=합격, `1`=불합격**이다 — 같은 `1`의 의미가 정반대로 뒤집힌다.
>
> **짚어주는 방법**: 데이터를 읽은 직후 `df["검사결과"].value_counts()`를 한 번 띄우고,
> "이 데이터에서 1은 불합격입니다"를 화면 한쪽에 적어둔 채로 수업을 시작한다.
> 4단계 정확도, 5단계 혼동행렬, 10차시 `불합격_확률`이 모두 이 약속 위에서 읽힌다는 점을 연결해준다.
>
> **학생이 자주 틀리는 지점**: 8차시 감각 그대로 `(y_test == 1).mean()`을 합격률로 쓰는 것 —
> 여기서는 그 값이 불합격률이다. 강의 슬라이드에도 `검사결과`(0=합격, 1=불합격)로 재고지돼 있으니
> 같은 표현으로 맞춰 읽어준다.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week09/week09_pass_fail_train.csv")
print(df.shape)
df = df.dropna()
print(df.shape)

(750, 9)
(735, 9)


## 2단계. Feature/Label 나누기
**설명 포인트**: 문자열 열(공정명, 설비번호)은 오늘 다루지 않는다는 것을 짚어준다.

In [2]:
feature_cols = ["온도_섭씨", "압력_Pa", "가스유량_slm", "두께_nm", "진공도_mTorr", "습도_pct"]
X = df[feature_cols]
y = df["검사결과"]

## 3단계. train/test 분리 및 모델 학습

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

## 4단계. 정확도 확인하기(그리고 기준선과 비교)
**가장 중요한 설명 포인트**: 모델 정확도(78.2%)가 기준선(78.9%)보다 낮다는 것을 직접 보여주며 '정확도의 함정'을 각인시킨다.

In [4]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)
baseline = (y_test == 0).mean()

print(f"모델 정확도: {accuracy:.1%}")
print(f"기준선 정확도: {baseline:.1%}")

모델 정확도: 78.2%
기준선 정확도: 78.9%


## 5단계. 혼동행렬 확인하기

In [5]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)

[[109   7]
 [ 25   6]]


## 6단계(종합). 중요 특징 확인하기
**설명 포인트**: 8차시 원인 후보(온도, 진공도, 두께)와 오늘 결과를 나란히 비교한다.

In [6]:
for name, importance in zip(feature_cols, model.feature_importances_):
    print(f"{name}: {importance:.1%}")

온도_섭씨: 51.7%
압력_Pa: 2.6%
가스유량_slm: 10.7%
두께_nm: 9.8%
진공도_mTorr: 20.8%
습도_pct: 4.4%


## 7단계. 오늘의 학습을 한 문장으로 정리하기
**진행 방법**: "정확도가 몇 %였나"가 아니라 "그 정확도를 믿어도 되나"를 묻는다. 4단계의 기준선과 5단계의 혼동행렬을 함께 언급하는 문장이 나오면 성공이다.
**설명 포인트**: 결과 해석 문장 쓰기는 1차시부터 10차시까지 매 차시 빠지지 않는 고정 활동이다(docs/curriculum.md 「차시 간 연결 원칙」). 코드를 친 것으로 끝내지 않고 자기 말로 바꿔 말해보게 하는 것이 이 과정의 목표 — "데이터로 공정 상태를 설명할 수 있는 사람" — 에 직접 닿는 활동이므로 시간이 모자라도 2~3분은 반드시 확보한다.

> (예시 답안) 정확도만 보면 모델이 괜찮아 보이지만, 혼동행렬을 보면 불합격을 잘 못 잡아낸다는 것을 알 수 있다. 불균형 데이터에서는 정확도만으로 판단하면 안 된다.

## 오류 대처 방법
- `ValueError` 발생 시: Feature에 문자열 열이 섞였는지 확인.
- 정확도가 비정상적으로 높으면: Feature에 검사결과가 실수로 포함됐는지 확인.

## 확장 실습(빠른 학습자용)
max_depth를 바꿔가며 정확도·혼동행렬 변화를 비교해보게 한다.

In [7]:
for depth in [2, 4, 8]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"max_depth={depth}: 정확도 {acc:.1%}")

max_depth=2: 정확도 78.9%
max_depth=4: 정확도 78.2%
max_depth=8: 정확도 75.5%


## 8단계. AI에게 질문하며 더 알아보기
**설명 포인트**: 불균형 데이터 대응법 질문이 나오면, 오늘 이미 확인한 "정확도의 함정"과
연결해 왜 다른 지표/방법이 필요한지 강조한다.

질문 예시:
- "결정트리 말고 다른 분류 모델에는 어떤 것들이 있나요?"
- "정확도 말고 모델 성능을 평가하는 다른 지표(정밀도, 재현율, F1)는 무엇을 의미하나요?"
- "max_depth를 너무 깊게 설정하면 어떤 문제(과적합)가 생기나요?"
- "합격이 압도적으로 많은 불균형 데이터를 다루는 다른 방법들이 있나요?"

## 9단계. AI에게 코드 생성 요청하고 직접 실행해보기
**설명 포인트**: `precision_score`, `recall_score`는 오늘 배우지 않은 함수다 — 혼동행렬의
네 칸(TP/FP/FN/TN)과 정밀도·재현율의 관계를 그림으로 짚어주면 이해가 빠르다.

프롬프트 예시: "실제값과 예측값 리스트를 받아서, 정확도·정밀도·재현율을 한 번에 출력하는
코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드와 실행 결과다.

In [8]:
from sklearn.metrics import precision_score, recall_score

print(f"정확도: {accuracy:.2f}")
print(f"정밀도: {precision_score(y_test, pred):.2f}")
print(f"재현율: {recall_score(y_test, pred):.2f}")

정확도: 0.78
정밀도: 0.46
재현율: 0.19


## 10단계. 연습문제 — 나무 깊이(max_depth) 바꿔보기
**설명 포인트**: `max_depth`가 작으면(2) 나무가 단순해 학습 데이터의 패턴도 충분히 못 잡을 수
있고(과소적합, underfitting), 너무 크면(8) 학습 데이터에 지나치게 맞춰져 새로운 데이터에서는
오히려 성능이 떨어질 수 있다(과대적합, overfitting)는 트레이드오프를 짚어준다. 아래 결과는
이 데이터·이 depth 값에서 실제로 관찰된 경향일 뿐 — depth나 데이터가 달라지면 순서가 바뀔 수도
있으므로 "depth가 크면 무조건 나쁘다"처럼 단정하지 않도록 주의한다.

In [9]:
model_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)
model_shallow.fit(X_train, y_train)
acc_shallow = accuracy_score(y_test, model_shallow.predict(X_test))

model_deep = DecisionTreeClassifier(max_depth=8, random_state=42)
model_deep.fit(X_train, y_train)
acc_deep = accuracy_score(y_test, model_deep.predict(X_test))

print(f"max_depth=2 정확도: {acc_shallow:.1%}")
print(f"max_depth=8 정확도: {acc_deep:.1%}")

max_depth=2 정확도: 78.9%
max_depth=8 정확도: 75.5%


## 11단계. AI로 재미있는 미니 프로그램 만들기 🎉 — 합격 예측 미니 데모
**설명 포인트**: 이미 학습된 `model`과 `feature_cols`를 그대로 재사용한다는 점을 강조한다 —
모델을 다시 학습시키지 않고도 새로운 값 하나만으로 바로 예측해볼 수 있다는 것이 오늘 만든
모델의 실용적인 쓰임새다. `검사결과`는 0=합격, 1=불합격으로 인코딩되어 있으므로
`if result == 0:` 분기가 핵심이다. 이 노트북은 자동 실행 검증 대상이라 `input()` 대신 고정된
샘플 값을 사용한다.

프롬프트 예시: "학습된 분류 모델(model)과 특징 이름 리스트(feature_cols)가 있을 때, 새로운 값
하나를 입력받아 데이터프레임 한 행으로 만들고 model.predict()로 합격/불합격을 재미있게
알려주는 파이썬 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드다.

In [10]:
sample = pd.DataFrame([{
    "온도_섭씨": 301.5, "압력_Pa": 1010.0, "가스유량_slm": 52.0,
    "두께_nm": 120.5, "진공도_mTorr": 5.0, "습도_pct": 45.0,
}])  # 실제로는 input()으로 각 값을 하나씩 입력받아도 좋습니다.

result = model.predict(sample[feature_cols])[0]
if result == 0:
    print("🎉 합격일 것 같아요!")
else:
    print("⚠️ 불합격 위험이 있어요, 공정을 다시 확인해보세요.")

🎉 합격일 것 같아요!
